# Beta 分布的极大似然估计（数值梯度）

> 对应代码：`MathStatsCode/code_in_notes/Chap.11/MLE_beta_nod.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebook/PyStata.md`。

**实验目的**：直接用 betaden() 写出对数似然而不提供梯度，由 ml 用数值方法求导完成估计。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.11`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。


In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.11")

import stata_setup
stata_setup.config("/opt/Stata", "mp")

清空数据并用 `set seed 8855` 固定种子。


In [ ]:
%%stata
clear
set seed 8855


**目标函数**

定义**不提供梯度**版本的似然程序：`args lnfj theta1 theta2` 是 lf 评估器的接口（只有似然、没有导数）；程序体只有一行 `replace lnfj = log(betaden(θ1,θ2,x))`——直接用 Stata 内置的 Beta 密度函数取对数作为对数似然。因为不给解析梯度，Stata 会用**数值微分**自动计算导数，代码写起来最简单，代价是计算更慢、精度略低。


In [ ]:
%%stata
cap program drop betaobj 
program betaobj 	
	args lnfj theta1 theta2
	quietly{ 		
		replace `lnfj'=log(betaden(`theta1',`theta2',x))
	} 
end 


**产生Beta分布数据**

生成数据：`set obs 1000`（1000个观测）、`gen x=rbeta(0.5,3)`（来自 Beta(0.5,3)）——与 MLE_beta.do 完全相同，便于对照。


In [ ]:
%%stata
set obs 1000 
gen x=rbeta(0.5,3)


**定义极大似然问题**

`ml model lf betaobj /alpha /beta`：用 lf（线性形式）评估器声明模型，两个参数都用 `/` 形式（表示只说系数、无自变量）。注意与 MLE_beta 的区别：这里没有 `freeparm`，也无需写命令行。


In [ ]:
%%stata
ml model lf betaobj /alpha /beta 


**寻找初始点（可选）**

`ml search`：让 Stata 自动尝试多组初始值，找到似然较高的起点后再交给优化算法——能有效避免局部极值。


In [ ]:
%%stata
ml search 


**最大化 并得到结果**

`ml maximize`：执行数值优化并输出估计结果（系数、标准误、置信区间、对数似然等），估计值应接近真值 (0.5, 3)。


In [ ]:
%%stata
ml maximize
